# Repairing Form Field Issues
This notebook show ways to detect and repair common structural issues of Form PDFs.

## Field-Widgets
A form field and a widget annotation are conceptually different objects.
* A widget annotation represents the visual aspect of a form field. It is shown on some page and referenced by the page's `/Annots` array.
* A Form Field contains all the non-appearance related properties like field name, field value, field type and several more. It visually represented by one ore more widget annotations - which need not exist all on the same page.
* A Form Field refers to its widgets via its `/Kids` array.
* A widget refers to its owning form field via its `/Parent` back-reference.

A "field-widget" is an object that contains all of the above information in one PDF object. This approachg may be comfortable, but there is a price to pay: hard to maintain, hard to disambiguate, programmatically cumbersome.

### Correction
PyMuPDF's `FormFields` class can detect and optionally repair field-widgets by dissolving each object in two separate objects: a field and a widget object.

In [1]:
import pymupdf
doc = pymupdf.open("field-widgets.pdf")

In [2]:
ff = doc.get_form_fields(verbose=True)  # do not repair, but tell us any problems

Found PDF specification issues:
   flat RBGs: 0
   flat fields: 0
   field-widgets: 6
Repair is recommended.



This file has 6 field-widgets. We could walk through the `FormFields` object and selectively invoke a repair, for instance:

In [3]:
for name, entry in ff.fields.items():
    if entry["is_widget"]:  # this is a field-widget!
        ff.repair_field_widget(name)
ff.refresh()  # reflect changes in the ff object
print(f"{ff.count_field_widgets} remaining field-widgets")

0 remaining field-widgets


The more comfortable approach is the conversion at `FormFields` construction time:

In [4]:
doc.close()  # recycle the document
doc = pymupdf.open(doc.name)

ff = doc.get_form_fields(repair=True, verbose=True)

Found PDF specification issues:
   flat RBGs: 0
   flat fields: 0
   field-widgets: 6
Repair has been requested ...
Repairs applied in 0.00 seconds.


## Flat Radio Button Groups ("flat RBG")
Here is PyMuPDF's concise definition of a ["flat RBG"](https://pymupdf.readthedocs.io/en/latest/glossary.html#glossary-flat-radio-button-group).

When switching between the On-Off states of radio buttons programmatically, flat RBGs pose a major challenge. This is because the buttons of the group have no structural connection to each other, like a common parent. The only thing that makes them a group is the common field name. Therefore, the Document must have been parsed in its entirety up front to identify lists of button widgets with ewaul names.

Similar to field-widgets discussed above, `FormFields` can detect and repair flat RBGs:

In [5]:
doc = pymupdf.open("flat-rbg.pdf")
ff = doc.get_form_fields(repair=True, verbose=True)

Found PDF specification issues:
   flat RBGs: 1
   flat fields: 0
   field-widgets: 0
Repair has been requested ...
Repairs applied in 0.00 seconds.


We can also work selectively here:

In [6]:
doc.close()
doc = pymupdf.open(doc.name)
ff = doc.get_form_fields()
for name, entry in ff.fields.items():
    if entry["rbg_type"] == "flat":  # this is a field-widget!
        ff.repair_flat_rbg(name)
ff.refresh()  # reflect changes in the ff object
print(f"{ff.count_flat_rbgs} remaining flat RBGs")

0 remaining flat RBGs
